# Bitcoin Trader Performance vs Market Sentiment
### PrimeTrade.ai — Data Science Assignment
**Submitted by:** [Your Name]  
**Date:** April 2025

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully!')

## 2. Load Datasets

In [ ]:
# Load the two datasets
fg = pd.read_csv('fear_greed_index.csv')
hd = pd.read_csv('historical_data.csv')

print('Fear & Greed Index shape:', fg.shape)
print('Historical Trades shape:', hd.shape)
fg.head()

In [ ]:
hd.head()

## 3. Data Cleaning & Preprocessing

In [ ]:
# Parse dates
fg['date'] = pd.to_datetime(fg['date'])
hd['date'] = pd.to_datetime(hd['Timestamp IST'], dayfirst=True, errors='coerce').dt.normalize()

# Check for nulls
print('Nulls in Fear & Greed:', fg.isnull().sum().sum())
print('Nulls in Historical Data:', hd.isnull().sum().sum())

print('\nFear & Greed date range:', fg['date'].min(), 'to', fg['date'].max())
print('Historical Data date range:', hd['date'].min(), 'to', hd['date'].max())

In [ ]:
# Merge datasets on date
df = hd.merge(fg[['date','value','classification']], on='date', how='left')
df = df.dropna(subset=['classification'])
print(f'Merged dataset shape: {df.shape}')
print('\nSentiment distribution in merged data:')
print(df['classification'].value_counts())

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics on Closed PnL
print('=== Closed PnL Statistics ===')
print(df['Closed PnL'].describe())

print('\n=== Unique Values ===')
print('Accounts:', df['Account'].nunique())
print('Coins traded:', df['Coin'].nunique())
print('Total trades:', len(df))

## 5. Core Analysis — Sentiment vs Performance

In [ ]:
# Define sentiment order and colors
order = ['Extreme Fear', 'Fear', 'Neutral', 'Greed', 'Extreme Greed']
palette = {
    'Extreme Fear': '#d62728',
    'Fear':         '#ff7f0e',
    'Neutral':      '#7f7f7f',
    'Greed':        '#2ca02c',
    'Extreme Greed':'#1a7a1a'
}

# Filter only closed trades (PnL != 0)
closed = df[df['Closed PnL'] != 0].copy()
closed['win'] = (closed['Closed PnL'] > 0).astype(int)

# Aggregate by sentiment
sent_stats = closed.groupby('classification').agg(
    avg_pnl   = ('Closed PnL', 'mean'),
    med_pnl   = ('Closed PnL', 'median'),
    win_rate  = ('win', 'mean'),
    trade_cnt = ('Closed PnL', 'count'),
    total_pnl = ('Closed PnL', 'sum')
).reindex(order)

print('=== Performance by Market Sentiment ===')
print(sent_stats.round(2))

## 6. Visualizations

In [ ]:
colors_ord = [palette[c] for c in order]

# Chart 1: Avg PnL by Sentiment
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor='#0d1117')

for ax in axes:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363d')

# Avg PnL
axes[0].bar(order, sent_stats['avg_pnl'], color=colors_ord, edgecolor='#30363d')
axes[0].axhline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)
axes[0].set_title('Avg PnL by Sentiment', color='white', fontweight='bold')
axes[0].set_xticklabels(order, rotation=20, ha='right', color='white')
axes[0].set_ylabel('Avg PnL (USD)', color='white')

# Win Rate
axes[1].bar(order, sent_stats['win_rate']*100, color=colors_ord, edgecolor='#30363d')
axes[1].axhline(50, color='yellow', linewidth=0.8, linestyle='--', alpha=0.7)
axes[1].set_title('Win Rate by Sentiment', color='white', fontweight='bold')
axes[1].set_xticklabels(order, rotation=20, ha='right', color='white')
axes[1].set_ylabel('Win Rate (%)', color='white')

# Trade Count
axes[2].bar(order, sent_stats['trade_cnt'], color=colors_ord, edgecolor='#30363d')
axes[2].set_title('Trade Volume by Sentiment', color='white', fontweight='bold')
axes[2].set_xticklabels(order, rotation=20, ha='right', color='white')
axes[2].set_ylabel('Number of Trades', color='white')

plt.tight_layout()
plt.savefig('charts_sentiment.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Chart saved!')

In [ ]:
# Chart 2: BUY vs SELL by sentiment
ls = closed.groupby(['classification','Side'])['Closed PnL'].mean().unstack(fill_value=0).reindex(order)

fig, ax = plt.subplots(figsize=(10, 5), facecolor='#0d1117')
ax.set_facecolor('#161b22')
x = np.arange(len(order))
w = 0.35
if 'BUY' in ls.columns:
    ax.bar(x - w/2, ls['BUY'], w, label='BUY', color='#58a6ff', edgecolor='#30363d')
if 'SELL' in ls.columns:
    ax.bar(x + w/2, ls['SELL'], w, label='SELL', color='#f78166', edgecolor='#30363d')
ax.set_xticks(x)
ax.set_xticklabels(order, rotation=20, ha='right', color='white')
ax.axhline(0, color='white', linewidth=0.6, linestyle='--', alpha=0.5)
ax.set_title('BUY vs SELL Avg PnL by Sentiment', color='white', fontweight='bold')
ax.set_ylabel('Avg PnL (USD)', color='white')
ax.legend(facecolor='#161b22', labelcolor='white')
for spine in ax.spines.values(): spine.set_edgecolor('#30363d')
ax.tick_params(colors='white')
plt.tight_layout()
plt.savefig('charts_buysell.png', dpi=120, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('BUY vs SELL chart saved!')

## 7. Key Insights & Recommendations

In [ ]:
print('='*60)
print('KEY INSIGHTS SUMMARY')
print('='*60)
print(f"""
1. EXTREME GREED PHASE:
   - Highest avg PnL: ${sent_stats.loc['Extreme Greed','avg_pnl']:.2f}/trade
   - Best win rate: {sent_stats.loc['Extreme Greed','win_rate']*100:.1f}%
   - Strategy: Momentum trading works best here

2. FEAR PHASE:
   - Strong avg PnL: ${sent_stats.loc['Fear','avg_pnl']:.2f}/trade  
   - Win rate: {sent_stats.loc['Fear','win_rate']*100:.1f}%
   - Strategy: Contrarian buying during dips is profitable

3. NEUTRAL PHASE:
   - Weakest returns: ${sent_stats.loc['Neutral','avg_pnl']:.2f}/trade
   - Strategy: Reduce position sizes during uncertainty

4. EXTREME FEAR:
   - Fewest trades: {int(sent_stats.loc['Extreme Fear','trade_cnt']):,}
   - Traders become cautious, less activity

5. BUY vs SELL:
   - BUY trades consistently outperform SELL across all sentiments
""")

## 8. Conclusion

This analysis reveals a strong relationship between Bitcoin market sentiment and trader performance:

- **Trade during Extreme Greed** for highest returns (momentum strategy)
- **Buy during Fear** for contrarian opportunities  
- **Reduce exposure during Neutral** sentiment — market direction is unclear
- **BUY side consistently outperforms SELL** across all sentiment categories
- Traders naturally reduce activity during Extreme Fear, which may be a rational risk management response

These findings can be used to build a **sentiment-driven trading strategy** that adjusts position sizing and direction based on the daily Fear & Greed Index reading.